# SMBHB Population Analysis & CGW SNR Diagnostics

This notebook analyzes SMBHB population outputs from the HPC pipeline and computes Continuous GW (CGW) SNR diagnostics.

## Quick Start

To run a complete CGW analysis:

1. **Activate the venv** (in terminal):
   ```bash
   . ~/jupyter/bin/activate
   cd /fred/oz005/users/bhoulden/SMBHB_population_injections
   export PYTHONPATH=$(pwd)
   jupyter lab
   ```

2. **Run cells in this order**:
   - Cell 1-2: **Setup** (imports, compatibility, plotting)
   - Cell 3-5: **Discovery & Loaders** (find result files, define unpickling)
   - Cell 6: **Load Binary Data** (populates `binary_df`)
   - Cell 7-8: **Simulation Summary** (aggregates to `sim_df`)
   - Cell 16-26: **CGW Analysis** (main plots: nearest distance, loudest binary, sky maps)

3. **Outputs**: Plots saved to `figures/` directory

## Formats Supported

- Old format: `data/YYYY-MM-DD/*/consistent_pop_synth*.pkl.gz`
- New Slurm format: `runs/YYYY-MM-DD_<scenario>/sim<NNN>/summary.pkl.gz` + `populations/subpop_*.pkl.gz`

Both are detected and loaded automatically.

In [47]:
from __future__ import annotations

import json
import gzip
import pickle
import re
import sys
from typing import Optional
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# NumPy compatibility: handle both NumPy 1.x (has numpy.core) and NumPy 2.x (has numpy._core)
NUMPY_VERSION_TUPLE = tuple(map(int, np.__version__.split('.')[:2]))
IS_NUMPY_2X = NUMPY_VERSION_TUPLE >= (2, 0)

if IS_NUMPY_2X:
    # We're on NumPy 2.x, create shim for unpickling NumPy 1.x pickles
    try:
        import numpy._core
        if not hasattr(np, 'core'):
            np.core = numpy._core
            sys.modules['numpy.core'] = numpy._core
    except (ImportError, AttributeError):
        pass

plt.style.use('default')
pd.set_option('display.max_columns', 50)

import types as _types

def _patch_numpy_modules():
    """Shim numpy.core → numpy._core for pickles created on NumPy 1.x."""
    if IS_NUMPY_2X:
        if 'numpy._core' not in sys.modules:
            _mod = _types.ModuleType('numpy._core')
            sys.modules['numpy._core'] = _mod
        if hasattr(np, '_core'):
            sys.modules['numpy._core'] = np._core
            if hasattr(np._core, 'multiarray'):
                sys.modules['numpy._core.multiarray'] = np._core.multiarray
    else:
        # NumPy 1.x: shim numpy._core → numpy.core
        if 'numpy._core' not in sys.modules:
            sys.modules['numpy._core'] = np.core
        if 'numpy._core.multiarray' not in sys.modules:
            sys.modules['numpy._core.multiarray'] = np.core.multiarray

_patch_numpy_modules()

SCENARIOS = ('optimistic', 'realistic', 'pessimistic')

In [48]:
import matplotlib as mpl

mpl.rcParams.update({
    # Font
    'font.family':       'serif',
    'font.serif':        ['Times New Roman', 'DejaVu Serif'],
    'mathtext.fontset':  'stix',        # matches Times for math/LaTeX symbols

    # Font sizes (ApJ single-column ~3.5in, so these scale well)
    'font.size':         12,
    'axes.titlesize':    12,
    'axes.labelsize':    12,
    'xtick.labelsize':   10,
    'ytick.labelsize':   10,
    'legend.fontsize':   10,

    # Figure size (single-column ApJ width = 3.5in)
    'figure.figsize':    (3.5, 2.8),    # use (7.0, 2.8) for double-column

    # Line/tick quality
    'axes.linewidth':    0.8,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.minor.width': 0.6,
    'ytick.minor.width': 0.6,
    'xtick.direction':   'in',          # ApJ style: ticks point inward
    'ytick.direction':   'in',
    'xtick.top':         True,          # ticks on all four sides
    'ytick.right':       True,
    'axes.spines.top':    True,
    'axes.spines.right':  True,
    'axes.spines.left':   True,
    'axes.spines.bottom': True,

    # Output
    'savefig.dpi':       300,
    'savefig.bbox':      'tight',
})

In [49]:
def infer_scenario(path: Path) -> str:
    lower_parts = [p.lower() for p in path.parts]
    for s in SCENARIOS:
        if any(s in p for p in lower_parts):
            return s
        if s in path.name.lower():
            return s
    return 'unknown'


def infer_run_id(path: Path) -> str:
    date_cfg_re = re.compile(r'^\d{4}-\d{2}-\d{2}_.+$')
    date_re = re.compile(r'^\d{4}-\d{2}-\d{2}$')
    sim_re = re.compile(r'^sim\d+$')

    for p in path.parts:
        if date_cfg_re.match(p):
            return p
    for p in path.parts:
        if date_re.match(p):
            return p

    for parent in path.parents:
        name = parent.name
        if not name:
            continue
        if name in {'data', 'runs', 'results', 'output', 'outputs'}:
            continue
        if sim_re.match(name):
            continue
        return name
    return 'legacy'


def discover_result_files(data_roots = Path('data')):
    # Summary-only mode: only load summary.pkl.gz files from runs/.
    if isinstance(data_roots, (str, Path)):
        roots = [Path(data_roots)]
    else:
        roots = [Path(root) for root in data_roots]

    patterns = ('summary.pkl.gz',)

    ordered_files: list[Path] = []
    seen: set[Path] = set()
    for root in roots:
        if not root.exists():
            continue
        root_matches = set()
        for pattern in patterns:
            root_matches.update(root.rglob(pattern))
        for path in sorted(p for p in root_matches if p.is_file()):
            if path not in seen:
                seen.add(path)
                ordered_files.append(path)

    return ordered_files


# ANALYSIS_ROOTS = [Path('data'), Path('runs')]
ANALYSIS_ROOTS = [Path('runs')]
result_files = discover_result_files(ANALYSIS_ROOTS)
print(f'Discovered {len(result_files)} summary files')
for p in result_files[::-1][:20]:  # Show most recent 20 files
    print('-', p)
if len(result_files) > 20:
    print('...')

Discovered 340 summary files
- runs/2026-05-21_pessimistic/sim000/summary.pkl.gz
- runs/2026-05-21_optimistic/sim354/summary.pkl.gz
- runs/2026-05-21_optimistic/sim353/summary.pkl.gz
- runs/2026-05-21_optimistic/sim352/summary.pkl.gz
- runs/2026-05-21_optimistic/sim349/summary.pkl.gz
- runs/2026-05-21_optimistic/sim348/summary.pkl.gz
- runs/2026-05-21_optimistic/sim347/summary.pkl.gz
- runs/2026-05-21_optimistic/sim346/summary.pkl.gz
- runs/2026-05-21_optimistic/sim345/summary.pkl.gz
- runs/2026-05-21_optimistic/sim344/summary.pkl.gz
- runs/2026-05-21_optimistic/sim343/summary.pkl.gz
- runs/2026-05-21_optimistic/sim341/summary.pkl.gz
- runs/2026-05-21_optimistic/sim340/summary.pkl.gz
- runs/2026-05-21_optimistic/sim339/summary.pkl.gz
- runs/2026-05-21_optimistic/sim338/summary.pkl.gz
- runs/2026-05-21_optimistic/sim337/summary.pkl.gz
- runs/2026-05-21_optimistic/sim336/summary.pkl.gz
- runs/2026-05-21_optimistic/sim335/summary.pkl.gz
- runs/2026-05-21_optimistic/sim334/summary.pkl.gz
-

In [50]:
def _extract_array_from_population_string(pop_text: str, key: str) -> Optional[np.ndarray]:
    # Extract key=array([ ... ]) from stringified PopulationArrays(...) output.
    pattern = rf"{re.escape(key)}=array\(\[(.*?)\]"
    match = re.search(pattern, pop_text, flags=re.DOTALL)
    if not match:
        return None

    raw = match.group(1).replace('\n', ' ')

    try:
        arr = np.fromstring(raw, sep=',')
    except ValueError:
        arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        try:
            arr = np.fromstring(raw.replace(',', ' '), sep=' ')
        except ValueError:
            arr = np.array([], dtype=float)

    if arr.size == 0 and raw.strip():
        tokens = re.findall(
            r'[-+]?(?:\d*\.\d+|\d+)(?:[eE][-+]?\d+)?|[-+]?inf|nan',
            raw,
            flags=re.IGNORECASE,
        )
        if tokens:
            arr = np.asarray([float(t) for t in tokens], dtype=float)

    return arr if arr.size > 0 else None


def _population_to_arrays(population_obj) -> Optional[dict[str, np.ndarray]]:
    # Case 0: Slurm summary payload written by stage2_inject.py.
    if isinstance(population_obj, dict):
        arrays = population_obj.get('arrays')
        if isinstance(arrays, dict):
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
            for k in ('global_idx', 'sim_id'):
                arr = arrays.get(k)
                if arr is not None:
                    out[k] = np.asarray(arr)
            return out if out else None

    # Case 1: population stored as dict with list/array values (compact representative format)
    if isinstance(population_obj, dict):
        has_pop_fields = any(k in population_obj for k in ('f', 'Mc', 'Mtot', 'D_comov', 'h0', 'z'))
        if has_pop_fields:
            out = {}
            for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'Mtot', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
                if k in population_obj:
                    arr = population_obj[k]
                    if arr is not None:
                        out[k] = np.asarray(arr, dtype=float)
            return out if out else None

    # Case 2: population stored as list of dict rows (legacy format)
    if isinstance(population_obj, list):
        if not population_obj:
            return None
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            vals = [row.get(k, np.nan) for row in population_obj]
            out[k] = np.asarray(vals, dtype=float)
        return out

    # Case 3: true PopulationArrays object (via pickle)
    if hasattr(population_obj, 'f') and hasattr(population_obj, 'Mc'):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'cgw_snr', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            if hasattr(population_obj, k):
                arr = getattr(population_obj, k)
                if arr is not None:
                    out[k] = np.asarray(arr, dtype=float)
        return out if out else None

    # Case 4: stringified PopulationArrays(...) (legacy JSON fallback)
    if isinstance(population_obj, str):
        out = {}
        for k in ('f', 'Mc', 'D', 'D_comov', 'h0', 'z', 'ra', 'dec', 'psi', 'iota', 'phi0'):
            arr = _extract_array_from_population_string(population_obj, k)
            if arr is not None:
                out[k] = arr.astype(float)
        return out if out else None

    return None


def _entry_to_binary_rows(entry: dict, source_file: Path, scenario: str, run_id: str, fallback_sim_indexx: int) -> list[dict]:
    arrays = _population_to_arrays(entry.get('population'))
    if arrays is None:
        return []

    d = arrays.get('D_comov')
    if d is None:
        d = arrays.get('D')

    n = len(arrays['f']) if 'f' in arrays else 0
    sim_index = entry.get('sim_index', fallback_sim_indexx)
    sim_index = int(sim_index) if isinstance(sim_index, (int, np.integer)) else fallback_sim_indexx

    rows = []
    for i in range(n):
        rows.append({
            'scenario': scenario,
            'run_id': run_id,
            'source_file': str(source_file),
            'sim_index': int(sim_index),
            'sim_key': sim_index,
            'binary_index': i,
            'f': float(arrays['f'][i]) if arrays.get('f') is not None else np.nan,
            'Mc': float(arrays['Mc'][i]) if arrays.get('Mc') is not None else np.nan,
            'D': float(d[i]) if d is not None else np.nan,
            'h0': float(arrays['h0'][i]) if arrays.get('h0') is not None else np.nan,
            'z': float(arrays['z'][i]) if arrays.get('z') is not None else np.nan,
            'cgw_snr': float(arrays['cgw_snr'][i]) if arrays.get('cgw_snr') is not None else np.nan,
            'ra': float(arrays['ra'][i]) if arrays.get('ra') is not None else np.nan,
            'dec': float(arrays['dec'][i]) if arrays.get('dec') is not None else np.nan,
            'psi': float(arrays['psi'][i]) if arrays.get('psi') is not None else np.nan,
            'iota': float(arrays['iota'][i]) if arrays.get('iota') is not None else np.nan,
            'phi0': float(arrays['phi0'][i]) if arrays.get('phi0') is not None else np.nan,
            'Mtot': float(arrays['Mtot'][i]) if arrays.get('Mtot') is not None else np.nan,
        })

        return rows

In [51]:
from dataclasses import dataclass, field

@dataclass
class PopulationArrays:
    f       : np.ndarray
    Mc      : np.ndarray
    Mtot    : np.ndarray
    D_comov : np.ndarray
    z       : np.ndarray
    h0      : np.ndarray
    ra      : np.ndarray
    dec     : np.ndarray
    psi     : np.ndarray
    iota    : np.ndarray
    phi0    : np.ndarray
    cgw_snr : np.ndarray
    amp_A   : Dict[str, np.ndarray] = field(default_factory=dict)
    amp_B   : Dict[str, np.ndarray] = field(default_factory=dict)

    def __len__(self): return len(self.f)

    def __getitem__(self, idx):
        new = PopulationArrays(
            f=self.f[idx], Mc=self.Mc[idx], Mtot=self.Mtot[idx],
            D_comov=self.D_comov[idx], z=self.z[idx], h0=self.h0[idx],
            ra=self.ra[idx], dec=self.dec[idx], psi=self.psi[idx],
            iota=self.iota[idx], phi0=self.phi0[idx], cgw_snr=self.cgw_snr[idx],
        )
        for k, v in self.amp_A.items(): new.amp_A[k] = v[idx]
        for k, v in self.amp_B.items(): new.amp_B[k] = v[idx]
        return new
    
from pathlib import Path

def _summary_rows_from_sim_directory(
    summary_file: Path,
    payload: dict,
    scenario: str,
    run_id: str,
    verbose: bool = False,
) -> pd.DataFrame:
    arrays = payload.get('arrays', {}) if isinstance(payload, dict) else {}
    if not isinstance(arrays, dict):
        return pd.DataFrame()

    global_idx = arrays.get('global_idx')
    if global_idx is None:
        if verbose:
            print(f'  No global_idx in summary: {summary_file}')
        return pd.DataFrame()

    global_idx = np.asarray(global_idx, dtype=np.int64)
    if global_idx.size == 0:
        return pd.DataFrame()

    FIELDS = ['f','Mc','Mtot','D_comov','z','h0','ra','dec','psi','iota','phi0','cgw_snr']

    # ── Fast path: physical arrays already in summary ──────────────────────
    if 'f' in arrays:
        if verbose:
            print(f'  Fast path: reading arrays directly from summary')
        def _coerce(name):
            if name in arrays:
                return np.asarray(arrays[name], dtype=float)
            return np.full(global_idx.size, np.nan)

        rows = []
        for i in range(global_idx.size):
            rows.append({
                'scenario': scenario, 'run_id': run_id,
                'source_file': str(summary_file),
                'sim_index': np.int32(-1),
                'binary_index': np.int32(i),
                'global_idx': np.int64(global_idx[i]),
                **{f: float(_coerce(f)[i]) for f in FIELDS},
                'D': float(_coerce('D_comov')[i]),
            })
        return pd.DataFrame(rows)

    # ── Fallback: read physical arrays from shard files ────────────────────
    pop_dir = summary_file.parent / 'populations'
    if not pop_dir.exists():
        if verbose:
            print(f'  No arrays in summary and no populations/ dir: {summary_file}')
        return pd.DataFrame()

    if verbose:
        print(f'  Sparse summary — reading from shards in {pop_dir}')

    # Build a map: global_idx → shard file + local position
    shard_files = sorted(pop_dir.glob('subpop_*.pkl.gz'))
    needed = set(global_idx.tolist())
    collected: dict[int, dict] = {}
    global_counter = 0

    for shard_path in shard_files:
        if not needed:
            break
        try:
            with gzip.open(shard_path, 'rb') as fh:
                pop = _CompatibilityUnpickler(fh).load()
        except Exception as e:
            if verbose:
                print(f'    Failed to load {shard_path.name}: {e}')
            global_counter += 0
            continue

        n = len(pop.f)
        for local_i in range(n):
            gidx = global_counter + local_i
            if gidx in needed:
                collected[gidx] = {
                    f: float(getattr(pop, f)[local_i])
                    for f in FIELDS
                    if hasattr(pop, f) and getattr(pop, f) is not None
                }
                needed.discard(gidx)
        global_counter += n
        del pop

    if verbose:
        print(f'  Recovered {len(collected)}/{global_idx.size} binaries from shards')

    # Extract sim_index from directory name
    sim_match = re.search(r'sim(\d+)', summary_file.parent.name)
    sim_index = int(sim_match.group(1)) if sim_match else -1

    rows = []
    for i, gidx in enumerate(global_idx.tolist()):
        row_data = collected.get(int(gidx), {})
        rows.append({
            'scenario': scenario, 'run_id': run_id,
            'source_file': str(summary_file),
            'sim_index': np.int32(sim_index),
            'binary_index': np.int32(i),
            'global_idx': np.int64(gidx),
            **{f: float(row_data.get(f, np.nan)) for f in FIELDS},
            'D': float(row_data.get('D_comov', np.nan)),
        })
    return pd.DataFrame(rows)

def _load_payload(path: Path, verbose: bool = False):
    """Load a .pkl.gz or .pkl file, trying CompatibilityUnpickler first."""
    import gzip, pickle
    try:
        with gzip.open(path, 'rb') as f:
            try:
                return _CompatibilityUnpickler(f).load()
            except Exception:
                pass
        with gzip.open(path, 'rb') as f:
            return pickle.load(f)
    except Exception as e:
        if verbose:
            print(f"  Failed to load {path}: {e}")
        return None
    
class _CompatibilityUnpickler(pickle.Unpickler):
    def find_class(self, module, name):
        # Redirect PopulationArrays from ANY module path to our local definition
        if name == 'PopulationArrays':
            return PopulationArrays
        # numpy core shim
        if module.startswith('numpy._core'):
            module = module.replace('numpy._core', 'numpy.core')
        elif module.startswith('numpy.core') and IS_NUMPY_2X:
            try:
                return super().find_class(module, name)
            except Exception:
                module = module.replace('numpy.core', 'numpy._core')
        # stub anything missing (numba etc.)
        try:
            return super().find_class(module, name)
        except Exception:
            _stub_missing_modules(module)
            try:
                return super().find_class(module, name)
            except Exception:
                return type(name, (), {'__module__': module})

In [52]:
# Override loader stubs for the Jupyter venv.
# The shard pickles import numba decorators such as `from numba import njit`.
# If we create `numba.njit` as a submodule, Python binds it as a module object
# and unpickling fails with `'module' object is not callable`.
# Keep the base `numba` module callable via decorator attributes, but do not
# create `numba.njit`/`numba.prange` submodules.

for _bad_mod in ('numba.njit', 'numba.prange', 'numba.vectorize', 'numba.guvectorize'):
    sys.modules.pop(_bad_mod, None)


def _stub_missing_modules(*module_names):
    for name in module_names:
        if name in sys.modules:
            continue
        try:
            __import__(name)
            continue
        except Exception:
            parts = name.split('.')
            if parts[0] == 'numba':
                stub = sys.modules.get('numba')
                if stub is None:
                    stub = types.ModuleType('numba')
                    stub.__path__ = []
                    sys.modules['numba'] = stub
                stub.jit = lambda *a, **k: (lambda f: f)
                stub.njit = lambda *a, **k: (lambda f: f)
                stub.vectorize = lambda *a, **k: (lambda f: f)
                stub.guvectorize = lambda *a, **k: (lambda f: f)
                stub.prange = range
                continue

            for i in range(len(parts)):
                parent = '.'.join(parts[: i + 1])
                if parent not in sys.modules:
                    stub = types.ModuleType(parent)
                    stub.__path__ = []
                    sys.modules[parent] = stub


# Re-run the numpy shim just in case this cell is executed standalone.
_patch_numpy_modules()

In [53]:
# DEBUG: Inspect what's in the discovered files
result_files_all = discover_result_files([Path('runs')])
print(f"Total files discovered: {len(result_files_all)}")

# Group by parent structure
from collections import defaultdict
by_sim = defaultdict(list)
for f in result_files_all:
    sim_dir = f.parent.name if 'sim' in f.parent.name else 'other'
    by_sim[sim_dir].append(f)

print(f"\nFile organization:")
for sim, files in sorted(by_sim.items())[:5]:
    print(f"  {sim}: {len(files)} files")
    for f in files[:2]:
        print(f"    - {f.name}")

# Try loading first few files and inspect their structure
print(f"\nInspecting file contents:")
for i, fp in enumerate(result_files_all[:3]):
    print(f"\n{i}. {fp.relative_to('.')}")
    try:
        payload = _load_payload(fp, verbose=False)
        print(f"   Type: {type(payload)}")
        if isinstance(payload, dict):
            print(f"   Keys: {list(payload.keys())[:10]}")
            if 'arrays' in payload:
                arrays = payload['arrays']
                print(f"   arrays type: {type(arrays)}")
                if isinstance(arrays, dict):
                    print(f"   arrays keys: {list(arrays.keys())}")
                    if 'global_idx' in arrays:
                        print(f"   global_idx: {type(arrays['global_idx'])}, len={len(arrays['global_idx']) if hasattr(arrays['global_idx'], '__len__') else '?'}")
        elif hasattr(payload, '__dict__'):
            print(f"   Attrs: {list(vars(payload).keys())[:5]}")
    except Exception as e:
        print(f"   Error: {type(e).__name__}: {e}")


Total files discovered: 340

File organization:
  sim000: 4 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim001: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim002: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim003: 2 files
    - summary.pkl.gz
    - summary.pkl.gz
  sim004: 2 files
    - summary.pkl.gz
    - summary.pkl.gz

Inspecting file contents:

0. runs/2026-05-20_optimistic/sim000/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['global_idx']
   global_idx: <class 'numpy.ndarray'>, len=223

1. runs/2026-05-20_optimistic/sim001/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['global_idx']
   global_idx: <class 'numpy.ndarray'>, len=213

2. runs/2026-05-20_optimistic/sim002/summary.pkl.gz
   Type: <class 'dict'>
   Keys: ['arrays', 'meta']
   arrays type: <class 'dict'>
   arrays keys: ['global_idx']
   global_idx: <class 'numpy

In [54]:
import gzip, pickle, traceback

path = 'runs/2026-05-20_optimistic/sim000/summary.pkl.gz'
try:
    with gzip.open(path, 'rb') as f:
        _CompatibilityUnpickler(f).load()
except Exception:
    traceback.print_exc()

In [55]:
# ============================================
# STEP 1: LOAD BINARY DATAFRAME FROM RUNS
# ============================================

def load_population_binary_table_robust(files, verbose=False):
    """
    Load binaries from summary files, handling multiple formats:
    1. Sparse summary: dict with 'arrays' key
    2. Compact format: dict with 'populations' list
    3. Direct array format: PopulationArrays-like object
    """
    frames = []

    for fp in files:
        if verbose:
            print(f"Loading: {fp.name}")

        try:
            scenario = infer_scenario(fp)
            run_id = infer_run_id(fp)
            payload = _load_payload(fp, verbose=False)

            if payload is None or (isinstance(payload, dict) and not payload):
                if verbose:
                    print("  → Empty payload")
                continue

            # FORMAT 1: Sparse summary (dict with 'arrays' key)
            if isinstance(payload, dict) and 'arrays' in payload:
                if verbose:
                    print("  → Sparse format detected")
                df_rows = _summary_rows_from_sim_directory(fp, payload, scenario, run_id, verbose=verbose)
                if not df_rows.empty:
                    frames.append(df_rows)
                    if verbose:
                        print(f"    → {len(df_rows)} rows from sparse")
                continue

            # FORMAT 2: Old compact format (dict with 'populations' list)
            if isinstance(payload, dict) and 'populations' in payload:
                pops = payload.get('populations', [])
                if isinstance(pops, list) and pops:
                    if verbose:
                        print(f"  → Compact format: {len(pops)} populations")
                    sim_match = re.search(r'sim(\\d+)', fp.parent.name)
                    sim_index = int(sim_match.group(1)) if sim_match else -1

                    rows = []
                    for pop_idx, pop_entry in enumerate(pops):
                        if not isinstance(pop_entry, dict):
                            continue
                        pop_obj = pop_entry.get('population', pop_entry)
                        arrays = _population_to_arrays(pop_obj)
                        if not arrays or 'f' not in arrays:
                            continue

                        n = len(arrays['f'])
                        for bi in range(n):
                            rows.append({
                                'scenario': scenario,
                                'run_id': run_id,
                                'source_file': str(fp),
                                'sim_index': np.int32(sim_index),
                                'binary_index': np.int32(bi),
                                'global_idx': np.int64(bi),
                                'f': np.float32(arrays['f'][bi]),
                                'Mc': np.float32(arrays.get('Mc', [np.nan] * n)[bi]),
                                'D': np.float32(arrays.get('D_comov', arrays.get('D', [np.nan] * n))[bi]),
                                'h0': np.float32(arrays.get('h0', [np.nan] * n)[bi]),
                                'z': np.float32(arrays.get('z', [np.nan] * n)[bi]),
                                'cgw_snr': np.float32(arrays.get('cgw_snr', [np.nan] * n)[bi]),
                                'ra': np.float32(arrays.get('ra', [np.nan] * n)[bi]),
                                'dec': np.float32(arrays.get('dec', [np.nan] * n)[bi]),
                                'psi': np.float32(arrays.get('psi', [np.nan] * n)[bi]),
                                'iota': np.float32(arrays.get('iota', [np.nan] * n)[bi]),
                                'phi0': np.float32(arrays.get('phi0', [np.nan] * n)[bi]),
                                'Mtot': np.float32(arrays.get('Mtot', [np.nan] * n)[bi]),
                            })

                    if rows:
                        frames.append(pd.DataFrame(rows))
                        if verbose:
                            print(f"    → {len(rows)} rows from compact")
                    continue

            # FORMAT 3: Direct array object (has f, Mc, D attributes)
            if hasattr(payload, 'f'):
                if verbose:
                    print(f"  → Direct array format: {type(payload).__name__}")
                sim_match = re.search(r'sim(\\d+)', fp.parent.name)
                sim_index = int(sim_match.group(1)) if sim_match else -1

                rows = []
                n = len(payload.f)
                for bi in range(n):
                    rows.append({
                        'scenario': scenario,
                        'run_id': run_id,
                        'source_file': str(fp),
                        'sim_index': np.int32(sim_index),
                        'binary_index': np.int32(bi),
                        'global_idx': np.int64(bi),
                        'f': np.float32(payload.f[bi]),
                        'Mc': np.float32(payload.Mc[bi] if hasattr(payload, 'Mc') else np.nan),
                        'D': np.float32(payload.D_comov[bi] if hasattr(payload, 'D_comov') else np.nan),
                        'h0': np.float32(payload.h0[bi] if hasattr(payload, 'h0') else np.nan),
                        'z': np.float32(payload.z[bi] if hasattr(payload, 'z') else np.nan),
                        'cgw_snr': np.float32(payload.cgw_snr[bi] if hasattr(payload, 'cgw_snr') else np.nan),
                        'ra': np.float32(payload.ra[bi] if hasattr(payload, 'ra') else np.nan),
                        'dec': np.float32(payload.dec[bi] if hasattr(payload, 'dec') else np.nan),
                        'psi': np.float32(payload.psi[bi] if hasattr(payload, 'psi') else np.nan),
                        'iota': np.float32(payload.iota[bi] if hasattr(payload, 'iota') else np.nan),
                        'phi0': np.float32(payload.phi0[bi] if hasattr(payload, 'phi0') else np.nan),
                        'Mtot': np.float32(payload.Mtot[bi] if hasattr(payload, 'Mtot') else np.nan),
                    })
                if rows:
                    frames.append(pd.DataFrame(rows))
                    if verbose:
                        print(f"    → {n} rows from direct arrays")
                continue

            if verbose:
                print(f"  → Unrecognized format: {type(payload)}")

        except Exception as e:
            if verbose:
                print(f"  Error: {type(e).__name__}: {e}")

    if not frames:
        if verbose:
            print("→ No rows loaded from any file")
        return pd.DataFrame()

    df = pd.concat(frames, ignore_index=True)
    if verbose:
        print(f"\n✓ Total: {len(df)} rows across {len(files)} files")
    return df


def _population_to_arrays(pop_obj):
    """Convert various population formats to a dict of arrays."""
    if pop_obj is None:
        return None

    if isinstance(pop_obj, dict) and 'f' in pop_obj:
        return pop_obj

    if hasattr(pop_obj, 'f'):
        return {
            'f': pop_obj.f,
            'Mc': getattr(pop_obj, 'Mc', None),
            'D_comov': getattr(pop_obj, 'D_comov', getattr(pop_obj, 'D', None)),
            'h0': getattr(pop_obj, 'h0', None),
            'z': getattr(pop_obj, 'z', None),
            'cgw_snr': getattr(pop_obj, 'cgw_snr', None),
            'ra': getattr(pop_obj, 'ra', None),
            'dec': getattr(pop_obj, 'dec', None),
            'psi': getattr(pop_obj, 'psi', None),
            'iota': getattr(pop_obj, 'iota', None),
            'phi0': getattr(pop_obj, 'phi0', None),
            'Mtot': getattr(pop_obj, 'Mtot', None),
        }

    if isinstance(pop_obj, str) and 'PopulationArrays' in pop_obj:
        try:
            import ast
            pass
        except Exception:
            pass

    return None


print("Step 1: Discovering result files...")
result_files = list(discover_result_files([Path('runs')]))
print(f"  → Found {len(result_files)} result files")

if result_files:
    print("\nStep 2: Loading population binary data with robust parser...")
    binary_df = load_population_binary_table_robust(result_files, verbose=True)
    print(f"\n  → Final: {len(binary_df)} binary rows loaded")
    if len(binary_df) > 0:
        print(f"  → Scenarios: {sorted(binary_df['scenario'].unique().tolist())}")
        if 'cgw_snr' in binary_df.columns:
            print(f"  → CGW SNR > 0: {(binary_df['cgw_snr'] > 0).sum()} binaries")
        sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
        if sky_cols:
            print(f"  → Sky-location columns loaded: {sky_cols}")
        display(binary_df.head())
else:
    print("ERROR: No result files found in runs/")
    binary_df = pd.DataFrame()

print(f"\nDataFrame shape: {binary_df.shape}")
print(f"Columns: {list(binary_df.columns) if not binary_df.empty else 'N/A'}")

Step 1: Discovering result files...
  → Found 340 result files

Step 2: Loading population binary data with robust parser...
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim000/populations
  Recovered 223/223 binaries from shards
    → 223 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim001/populations
  Recovered 213/213 binaries from shards
    → 213 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim002/populations
  Recovered 226/226 binaries from shards
    → 226 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim003/populations
  Recovered 225/225 binaries from shards
    → 225 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detec

,scenario,run_id,source_file,sim_index,binary_index,global_idx,f,Mc,Mtot,D_comov,z,h0,ra,dec,psi,iota,phi0,cgw_snr,D
0,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,0,0,3.740068e-09,3.046477e+09,7.129068e+09,11223.199219,19.999960,8.093844e-16,0.610840,0.614258,0.498047,2.998047,4.625000,0.632973,11223.199219
1,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,1,1,2.203749e-09,4.027252e+09,1.050541e+10,11223.199219,19.999960,5.805208e-16,0.039093,0.642090,2.138672,2.173828,4.875000,0.186351,11223.199219
2,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,2,2,2.022618e-09,2.169020e+09,5.023037e+09,11223.199219,19.999960,2.017995e-16,5.464844,-0.038727,0.497803,2.824219,2.996094,0.116122,11223.199219
3,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,3,3,2.389972e-09,2.846012e+09,6.544322e+09,11223.199219,19.999960,4.389251e-16,3.919922,0.323730,2.146484,2.269531,0.971680,0.187614,11223.199219
4,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,4,4,2.209112e-09,2.515746e+09,5.782557e+09,2724.542725,0.735521,4.487677e-16,4.035156,-0.986816,0.023621,2.244141,4.750000,0.154709,2724.542725



DataFrame shape: (76571, 19)
Columns: ['scenario', 'run_id', 'source_file', 'sim_index', 'binary_index', 'global_idx', 'f', 'Mc', 'Mtot', 'D_comov', 'z', 'h0', 'ra', 'dec', 'psi', 'iota', 'phi0', 'cgw_snr', 'D']


In [58]:
# ============================================
# STEP 1B: CGW HELPER FUNCTIONS
# ============================================

def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build per-simulation CGW summary table from the loaded binary dataframe."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    df_cgw = binary_df[binary_df['cgw_snr'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in df_cgw.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        nearest_d = group['D'].min() if 'D' in group.columns else np.nan
        loudest_idx = group['cgw_snr'].idxmax()
        loudest = group.loc[loudest_idx]

        records.append({
            'scenario':        scenario,
            'run_id':          run_id,
            'source_file':     str(loudest.get('source_file', '')),
            'sim_index':       int(sim_index),
            'sim_key':         f"{scenario}:{run_id}:{sim_index}",
            'nearest_D':       float(nearest_d),
            'loudest_cgw_snr': float(loudest['cgw_snr']),
            'loudest_h0':      float(loudest.get('h0', np.nan)),
            'loudest_Mc':      float(loudest.get('Mc', np.nan)),
            'loudest_D':       float(loudest.get('D', np.nan)),
            'loudest_f':       float(loudest.get('f', np.nan)),
            'loudest_ra':      float(loudest.get('ra',   np.nan)) if pd.notna(loudest.get('ra'))   else np.nan,
            'loudest_dec':     float(loudest.get('dec',  np.nan)) if pd.notna(loudest.get('dec'))  else np.nan,
            'loudest_psi':     float(loudest.get('psi',  np.nan)) if pd.notna(loudest.get('psi'))  else np.nan,
            'loudest_iota':    float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
            'loudest_phi0':    float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in binary_df.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'has_cgw': bool((group['cgw_snr'] > 0).any()),
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()

In [59]:
# ============================================
# STEP 2: BUILD CGW ANALYSIS TABLE
# ============================================

print("Building CGW analysis table...")

# Always reload from the summary files so Step 2 cannot use stale binary_df state.
binary_df = load_population_binary_table_robust(result_files, verbose=True)
print(f"\n  → Reloaded: {len(binary_df)} binary rows loaded from summary.pkl.gz files")
if not binary_df.empty:
    print(f"  → Columns: {list(binary_df.columns)}")
    if 'cgw_snr' in binary_df.columns:
        finite_cgw = binary_df['cgw_snr'].replace([np.inf, -np.inf], np.nan).dropna()
        print(f"  → cgw_snr present: {len(finite_cgw)} finite values, {int((binary_df['cgw_snr'] > 0).sum())} positive")
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns: {sky_cols if sky_cols else 'none'}")

# Build tables
cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")

    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_f', 'loudest_ra', 'loudest_dec', 'loudest_psi', 'loudest_iota', 'loudest_phi0']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")

Building CGW analysis table...
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim000/populations
  Recovered 223/223 binaries from shards
    → 223 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim001/populations
  Recovered 213/213 binaries from shards
    → 213 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim002/populations
  Recovered 226/226 binaries from shards
    → 226 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim003/populations
  Recovered 225/225 binaries from shards
    → 225 rows from sparse
Loading: summary.pkl.gz
  → Sparse format detected
  Sparse summary — reading from shards in runs/2026-05-20_optimistic/sim004/populations
  

,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,338,3.465343,1508.310059,58.442511
1,pessimistic,1,2.562823,2.562823,16.273783
2,realistic,1,2.394356,2.394356,7.335451



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_f,loudest_ra,loudest_dec,loudest_psi,loudest_iota,loudest_phi0
0,optimistic,0,6.535964,5.819182e-15,1.585873e+09,4.420853e-09,6.093750,0.940918,2.691406,2.783203,1.345703
1,optimistic,1,2.879694,1.589647e-15,3.988603e+09,6.598508e-09,4.230469,0.575684,2.050781,3.117188,1.360352
2,optimistic,2,4.502238,2.139610e-15,5.157685e+09,2.285923e-08,5.480469,-0.077515,0.490234,3.037109,2.160156
3,optimistic,3,9.250917,9.982745e-16,4.109262e+09,4.999964e-09,5.402344,0.333740,0.145020,0.200562,2.697266
4,optimistic,4,2.408288,1.295310e-15,4.321133e+09,1.561805e-08,4.847656,0.026810,0.314209,0.813965,5.707031
5,optimistic,5,9.200004,3.752163e-16,2.315347e+09,3.782202e-09,4.281250,-0.389160,1.678711,0.564941,6.125000
6,optimistic,7,7.603668,1.879759e-16,1.335515e+09,2.486693e-09,4.167969,-0.545410,2.376953,1.773438,5.046875
7,optimistic,8,1.021972,1.349843e-15,1.077665e+10,6.902761e-09,4.355469,0.602051,3.017578,1.634766,5.589844
8,optimistic,9,2.155031,1.118092e-15,6.032545e+09,1.110674e-08,3.714844,-0.420898,0.894043,0.179565,3.511719
9,optimistic,10,1.361115,8.724552e-16,6.785480e+09,6.735598e-09,3.750000,-0.722656,2.990234,0.222412,5.578125


In [60]:
# ============================================
# STEP 3: PLOT CGW ANALYSIS RESULTS  
# ============================================

if cgw_sim_df.empty:
    print("Skipping plots: No CGW data available")
else:
    from pathlib import Path
    Path('figures').mkdir(exist_ok=True)
    
    print("\nGenerating plots...")
    
    # Helper: extract values by scenario
    def _extract(df, col):
        return {
            s: df.loc[df['scenario'] == s, col].to_numpy()
            for s in SCENARIOS
        }
    
    # 1. Nearest distance
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['nearest_D'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Nearest SMBHB comoving distance [Mpc]')
    ax.set_ylabel('Count')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig("figures/cgw_nearest_SMBHB.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_nearest_SMBHB.pdf")
    
    # 2. CGW SNR distribution
    fig, ax = plt.subplots(figsize=(9, 4.8))
    for scenario in SCENARIOS:
        sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario]['loudest_cgw_snr'].dropna()
        if len(sub) > 0:
            ax.hist(sub, bins=30, alpha=0.5, label=scenario, density=False)
    ax.set_xscale('log')
    ax.set_xlabel('Loudest binary CGW SNR')
    ax.set_ylabel('Count')
    ax.axvline(5.0, color='red', linestyle='--', linewidth=2, label='SNR=5 threshold')
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig("figures/cgw_snr_distribution.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_snr_distribution.pdf")
    
    # 3. Loudest binary parameters
    fig, axes = plt.subplots(2, 2, figsize=(13, 8))
    plot_cfg = [
        ('loudest_h0', '$h_0$', True),
        ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]', True),
        ('loudest_D', r'$D_{\rm{comov}}$ [Mpc]', True),
        ('loudest_f', '$f$ [Hz]', True),
    ]
    for ax, (col, label, logx) in zip(axes.ravel(), plot_cfg):
        for scenario in SCENARIOS:
            sub = cgw_sim_df[cgw_sim_df['scenario'] == scenario][col].dropna()
            if len(sub) > 0:
                ax.hist(sub, bins=20, alpha=0.5, label=scenario, density=False)
        ax.set_xlabel(label)
        ax.set_ylabel('Count')
        if logx:
            ax.set_xscale('log')
        ax.legend(frameon=False)
    fig.suptitle('Loudest Binary (max CGW SNR) Properties', fontsize=14, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/cgw_loudest_binary_parameters.pdf", dpi=300, bbox_inches='tight')
    print("  ✓ figures/cgw_loudest_binary_parameters.pdf")
    
    print("\n✓ All plots saved to figures/")
    print("  - cgw_nearest_SMBHB.pdf")
    print("  - cgw_snr_distribution.pdf")
    print("  - cgw_loudest_binary_parameters.pdf")


Generating plots...
  ✓ figures/cgw_nearest_SMBHB.pdf
  ✓ figures/cgw_snr_distribution.pdf
  ✓ figures/cgw_loudest_binary_parameters.pdf

✓ All plots saved to figures/
  - cgw_nearest_SMBHB.pdf
  - cgw_snr_distribution.pdf
  - cgw_loudest_binary_parameters.pdf


In [61]:
import sys, types
import numpy as np

# Create compatibility modules that older pickles expect.
if 'numpy._core' not in sys.modules:
    mod_core = types.ModuleType('numpy._core')
    sys.modules['numpy._core'] = mod_core

# Point numpy._core.multiarray to the real implementation
sys.modules['numpy._core.multiarray'] = np.core.multiarray
setattr(sys.modules['numpy._core'], 'multiarray', np.core.multiarray)

# Also ensure attribute aliasing some pickles use
if not hasattr(np.core.multiarray, '_reconstruct') and hasattr(np.core.multiarray, 'reconstruct'):
    setattr(np.core.multiarray, '_reconstruct', getattr(np.core.multiarray, 'reconstruct'))

In [62]:
import gzip, pickle
p = Path('runs/2026-05-20_optimistic/sim000/summary.pkl.gz')
payload = pickle.load(gzip.open(p,'rb'))
print(payload.keys())
print(sorted(payload.get('arrays', {}).keys()))

dict_keys(['arrays', 'meta'])
['global_idx']


In [63]:
def hist_by_scenario(data_by_scenario: dict[str, np.ndarray], bins, xlabel: str, title: str, save = False, savename = None, logx: bool = False):
    fig, ax = plt.subplots(figsize=(9, 5))
    scenario_labels = (r"$p(M) \propto M^{-1.21}$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{10}\;\mathrm{M}_{\odot}})$", 
         r"$p(M) \propto M^{-1.21}\exp(-\frac{M}{10^{9}\;\mathrm{M}_{\odot}})$")
    for (scenario, label) in zip(SCENARIOS, scenario_labels):
        vals = data_by_scenario.get(scenario)
        if vals is None or len(vals) == 0:
            continue
        ax.hist(vals, bins=bins, alpha=0.45, density=False, label=label)

    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('PDF')
    # ax.set_title(title)
    ax.legend(frameon=False)
    plt.tight_layout()
    if save:
        plt.savefig(savename)
    return fig, ax



nearest_by_scenario = {
    s: sim_df.loc[sim_df['scenario'] == s, 'nearest_D'].dropna().to_numpy()
    for s in SCENARIOS
}

hist_by_scenario(
    nearest_by_scenario,
    bins=list(np.logspace(0, 4, 50)),
    xlabel=r'$D_{\rm{comov}}$',
    title='Nearest Binary Per Simulation by Scenario',
    logx=True,
    save = True,
    savename="data/2026-04-01/nearest_SMBHB.pdf"
)


(<Figure size 900x500 with 1 Axes>,
 <AxesSubplot:xlabel='$D_{\\rm{comov}}$', ylabel='PDF'>)

In [64]:
loudest_by_scenario = {
    s: sim_df.loc[sim_df['scenario'] == s, 'loudest_h0'].dropna().to_numpy()
    for s in SCENARIOS
}

_ = hist_by_scenario(
    loudest_by_scenario,
    bins=list(np.logspace(-15, -12, 50)),
    xlabel='Maximum h0 per simulation',
    title='Loudest Binary Proxy Per Simulation by Scenario',
    logx=True,
    save = True,
    savename="data/2026-04-01/biggest_h0_SMBHB.pdf"
)

In [65]:
def add_distribution_weights(df: pd.DataFrame, mode: str = 'all_binaries') -> pd.DataFrame:
    out = df.copy()
    if mode == 'all_binaries':
        out['weight'] = 1.0
        return out

    if mode == 'per_sim_equal':
        counts = out.groupby('sim_index')['binary_index'].transform('count').astype(float)
        out['weight'] = 1.0 / counts
        return out

    raise ValueError("mode must be 'all_binaries' or 'per_sim_equal'")


weight_mode = 'per_sim_equal'  # change to 'per_sim_equal' if desired
dist_df = add_distribution_weights(binary_df, mode=weight_mode)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_specs = [
    ('Mc', 'Chirp mass Mc', True),
    ('f', 'GW frequency f [Hz]', True),
    ('D', 'Comoving distance D', True),
]
bins_mass = list(np.logspace(6, 11, 50))
bins_f = list(np.logspace(-9, -6, 50))
bins_dist = list(np.logspace(1, 5, 50))
bins = [bins_mass, bins_f, bins_dist]


for ax, (col, xlabel, logx), _bin in zip(axes, plot_specs, bins):
    for scenario in SCENARIOS:
        sub = dist_df[(dist_df['scenario'] == scenario) & np.isfinite(dist_df[col])]
        if sub.empty:
            continue
        ax.hist(
            sub[col].to_numpy(),
            bins=_bin,
            weights=sub['weight'].to_numpy(),
            density=True,
            alpha=0.35,
            label=scenario,
        )
    if logx:
        ax.set_xscale('log')
    ax.set_xlabel(xlabel)
    ax.set_ylabel('Density')
    ax.set_xlim(min(_bin), max(_bin))

axes[0].legend(frameon=False)
fig.suptitle(f'Population Distributions by Scenario (weight_mode={weight_mode})')
plt.tight_layout()
plt.savefig("data/2026-04-01/distributions.pdf")

In [66]:
summary = (
    sim_df.groupby('scenario', as_index=False)
    .agg(
        n_sims=('sim_index', 'nunique'),
        nearest_D_median=('nearest_D', 'median'),
        nearest_D_p10=('nearest_D', lambda x: np.nanpercentile(x, 10)),
        nearest_D_p90=('nearest_D', lambda x: np.nanpercentile(x, 90)),
        loudest_h0_median=('loudest_h0', 'median'),
        loudest_h0_p10=('loudest_h0', lambda x: np.nanpercentile(x, 10)),
        loudest_h0_p90=('loudest_h0', lambda x: np.nanpercentile(x, 90)),
    )
)

summary

,scenario,n_sims,nearest_D_median,nearest_D_p10,nearest_D_p90,loudest_h0_median,loudest_h0_p10,loudest_h0_p90
0,optimistic,303,58.442511,11.036899,187.646738,1.469839e-15,1.556521e-16,4.427995e-15
1,pessimistic,1,16.273783,16.273783,16.273783,1.622962e-15,1.622962e-15,1.622962e-15
2,realistic,1,7.335451,7.335451,7.335451,1.720132e-15,1.720132e-15,1.720132e-15


## Notes on Combining Across Simulations

Default in this notebook:
- nearest/loudest metrics are per simulation (one point per sim), which is robust for scenario comparison
- distribution plots use all binaries by default (`weight_mode='all_binaries'`)

Alternative:
- set `weight_mode='per_sim_equal'` to avoid simulations with more binaries dominating parameter distributions

If you later add SNR contribution diagnostics, keep the same grouping key (`sim_id`) so all summaries stay aligned.

## Storage Recommendation

For your workflow, use one primary format by default: **compressed pickle (`.pkl.gz`)**.

Why this is the best default here:
- full-fidelity recovery of `PopulationArrays` objects
- much smaller than plain `.pkl`
- still straightforward to load in Python

When to additionally save `.npz`:
- only if disk size or data transfer is a bottleneck
- only for plotting/array workflows (not full object reconstruction)

In [70]:
# Minimal examples
from pathlib import Path

from utils import load_results_pickle_gz, save_results_compact_npz

# 1) Preferred: load full-fidelity compressed pickle
pkl_gz_path = Path('data/2026-03-31/realistic/consistent_population_realistic_targetSNR4.0_sims10.pkl.gz')
if pkl_gz_path.exists():
    results_obj = load_results_pickle_gz(pkl_gz_path)
    print('Loaded:', pkl_gz_path)
    print('n sims:', len(results_obj.get('populations', [])))
else:
    print('Example file not found:', pkl_gz_path)

# 2) Optional: create compact NPZ for plotting-only pipelines
if 'results_obj' in locals():
    npz_path = pkl_gz_path.with_suffix('').with_suffix('.npz')
    save_results_compact_npz(results_obj, npz_path)
    print('Saved compact npz:', npz_path)

Example file not found: data/2026-03-31/realistic/consistent_population_realistic_targetSNR4.0_sims10.pkl.gz


## Continuous GW (CGW) SNR Diagnostics

This section reads CGW diagnostics saved in the same `compact_results` structure used in `main.py`, i.e. per simulation:
- `compact_results['populations'][i]['population']`
- `compact_results['populations'][i]['cgw_analysis']['top_sources']`

It builds per-simulation metrics across scenarios and then plots:
- nearest SMBHB distance per simulation,
- loudest-by-CGW binary (`max SNR`) per simulation: `h0`, `Mc`, `D`, and `f`,
- two sky maps for loudest-by-CGW binaries: plain dots and SNR-colored.


In [67]:
len(result_files_CGW)

340

In [68]:
# Simplified CGW analysis: Build table directly from binary_df (works for summary.pkl.gz payloads)

def build_cgw_simulation_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Build CGW simulation table from the loaded binary dataframe."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    df_cgw = binary_df[binary_df['cgw_snr'] > 0].copy()
    if df_cgw.empty:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in df_cgw.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        nearest_d = group['D'].min() if 'D' in group.columns else np.nan
        loudest_idx = group['cgw_snr'].idxmax()
        loudest = group.loc[loudest_idx]

        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'source_file': str(loudest.get('source_file', '')),
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'nearest_D': float(nearest_d),
            'loudest_cgw_snr': float(loudest['cgw_snr']),
            'loudest_h0': float(loudest.get('h0', np.nan)),
            'loudest_Mc': float(loudest.get('Mc', np.nan)),
            'loudest_D': float(loudest.get('D', np.nan)),
            'loudest_f': float(loudest.get('f', np.nan)),
            'loudest_ra': float(loudest.get('ra', np.nan)) if pd.notna(loudest.get('ra')) else np.nan,
            'loudest_dec': float(loudest.get('dec', np.nan)) if pd.notna(loudest.get('dec')) else np.nan,
            'loudest_psi': float(loudest.get('psi', np.nan)) if pd.notna(loudest.get('psi')) else np.nan,
            'loudest_iota': float(loudest.get('iota', np.nan)) if pd.notna(loudest.get('iota')) else np.nan,
            'loudest_phi0': float(loudest.get('phi0', np.nan)) if pd.notna(loudest.get('phi0')) else np.nan,
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


def build_cgw_sim_inventory_table_simple(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Track whether each simulation has any nonzero CGW SNR binaries."""
    if binary_df.empty or 'cgw_snr' not in binary_df.columns:
        return pd.DataFrame()

    records = []
    for (scenario, run_id, sim_index), group in binary_df.groupby(['scenario', 'run_id', 'sim_index'], dropna=False):
        records.append({
            'scenario': scenario,
            'run_id': run_id,
            'sim_index': int(sim_index),
            'sim_key': f"{scenario}:{run_id}:{sim_index}",
            'has_cgw': bool((group['cgw_snr'] > 0).any()),
        })

    return pd.DataFrame.from_records(records) if records else pd.DataFrame()


# Build tables from summary.pkl.gz-backed rows
print('Reloading binary_df from summary.pkl.gz files...')
binary_df = load_population_binary_table_robust(result_files, verbose=False)
print(f'  → Reloaded {len(binary_df)} binary rows')
if not binary_df.empty:
    sky_cols = [c for c in ('ra', 'dec', 'psi', 'iota', 'phi0') if c in binary_df.columns]
    print(f"  → Sky-location columns loaded: {sky_cols if sky_cols else 'none'}")

cgw_sim_inventory_df = build_cgw_sim_inventory_table_simple(binary_df)
cgw_sim_df = build_cgw_simulation_table_simple(binary_df)
# Keep generic aliases for later cells that still expect the old names.
sim_df = cgw_sim_df.copy()
result_files_CGW = result_files.copy()

if not cgw_sim_df.empty:
    print(f"\n✓ CGW simulations found: {len(cgw_sim_df)}")
    print(f"  Scenarios: {sorted(cgw_sim_df['scenario'].unique().tolist())}")
    
    summary_cgw = (
        cgw_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_cgw_snr=('loudest_cgw_snr', 'median'),
            max_cgw_snr=('loudest_cgw_snr', 'max'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    print("\nCGW Summary by Scenario:")
    display(summary_cgw)
    print("\nFirst 10 CGW sources:")
    display(cgw_sim_df[['scenario', 'sim_index', 'loudest_cgw_snr', 'loudest_h0', 'loudest_Mc', 'loudest_f', 'loudest_ra', 'loudest_dec']].head(10))
else:
    print("✗ No CGW data found in summary.pkl.gz files (all cgw_snr ≤ 0, or the summary payload does not include cgw_snr/sky-location arrays)")

Reloading binary_df from summary.pkl.gz files...
  → Reloaded 76571 binary rows
  → Sky-location columns loaded: ['ra', 'dec', 'psi', 'iota', 'phi0']

✓ CGW simulations found: 340
  Scenarios: ['optimistic', 'pessimistic', 'realistic']

CGW Summary by Scenario:


,scenario,n_sims,median_cgw_snr,max_cgw_snr,median_nearest_D
0,optimistic,338,3.465343,1508.310059,58.442511
1,pessimistic,1,2.562823,2.562823,16.273783
2,realistic,1,2.394356,2.394356,7.335451



First 10 CGW sources:


,scenario,sim_index,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_f,loudest_ra,loudest_dec
0,optimistic,0,6.535964,5.819182e-15,1.585873e+09,4.420853e-09,6.093750,0.940918
1,optimistic,1,2.879694,1.589647e-15,3.988603e+09,6.598508e-09,4.230469,0.575684
2,optimistic,2,4.502238,2.139610e-15,5.157685e+09,2.285923e-08,5.480469,-0.077515
3,optimistic,3,9.250917,9.982745e-16,4.109262e+09,4.999964e-09,5.402344,0.333740
4,optimistic,4,2.408288,1.295310e-15,4.321133e+09,1.561805e-08,4.847656,0.026810
5,optimistic,5,9.200004,3.752163e-16,2.315347e+09,3.782202e-09,4.281250,-0.389160
6,optimistic,7,7.603668,1.879759e-16,1.335515e+09,2.486693e-09,4.167969,-0.545410
7,optimistic,8,1.021972,1.349843e-15,1.077665e+10,6.902761e-09,4.355469,0.602051
8,optimistic,9,2.155031,1.118092e-15,6.032545e+09,1.110674e-08,3.714844,-0.420898
9,optimistic,10,1.361115,8.724552e-16,6.785480e+09,6.735598e-09,3.750000,-0.722656


In [69]:
# Okabe-Ito colourblind-safe palette
scenario_labels = (r"$p(M) \propto M^{-1.21}\exp(-M/10^{9}\;\mathrm{M}_{\odot})$", 
         r"$p(M) \propto M^{-1.21}\exp(-M/10^{8.5}\;\mathrm{M}_{\odot})$", 
         r"$p(M) \propto M^{-1.21}\exp(-M/10^{8}\;\mathrm{M}_{\odot})$")


scenario_styles = {
    r"$p(M) \propto M^{-1.21}\exp(-M/10^{9}\;\mathrm{M}_{\odot})$" :  {'color': '#0072B2', 'linestyle': '-',  'linewidth': 2.2},
    r"$p(M) \propto M^{-1.21}\exp(-M/10^{8.5}\;\mathrm{M}_{\odot})$" :   {'color': '#E69F00', 'linestyle': '--', 'linewidth': 2.2},
    r"$p(M) \propto M^{-1.21}\exp(-M/10^{8}\;\mathrm{M}_{\odot})$" : {'color': '#D55E00', 'linestyle': ':',  'linewidth': 2.5},
}


In [70]:
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from scipy.stats import gaussian_kde
import numpy as np

scenario_order = [s for s in SCENARIOS if s in set(cgw_sim_df.get('scenario', []))]

# Map original scenario names -> LaTeX labels (order must match SCENARIOS order)
scenario_label_map = {
    s: lab for s, lab in zip(scenario_order, scenario_labels)
}

scenario_styles = {
    s: style for s, style in zip(scenario_order, [
        {'color': 'lime', 'linestyle': '-',  'linewidth': 2.2},
        {'color': 'violet', 'linestyle': '--', 'linewidth': 2.2},
        {'color': 'navy', 'linestyle': ':',  'linewidth': 2.5},
    ])
}

def _scenario_histogram(ax, scenario_vals: dict, xlabel: str, logx: bool = False, show_legend: bool = True, vline: float = None, loc: str = 'best', bbox_anch_coords=None):
    all_vals = np.concatenate([v for v in scenario_vals.values() if len(v) > 0])
    all_vals = all_vals[np.isfinite(all_vals)]
    if len(all_vals) == 0:
        return
    if logx:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), 25)

    for s in scenario_order:
        vals = scenario_vals.get(s, np.array([]))
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        style = scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})
        ax.hist(
            vals,
            bins=bins,
            histtype='step',
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
            label=s,
            density=False,
        )

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Count', fontsize=11)
    ax.tick_params(labelsize=9)
    ax.spines['top'].set_visible(True)
    ax.spines['right'].set_visible(True)
    if logx:
        ax.set_xscale('log')
    if vline is not None:
        ax.axvline(vline, color='black', linestyle='--', linewidth=1.5)
    if show_legend:
        handles = [
            mlines.Line2D(
                [], [],
                color=scenario_styles[s]['color'],
                linestyle=scenario_styles[s]['linestyle'],
                linewidth=scenario_styles[s]['linewidth'],
                label=scenario_label_map[s],
            )
            for s in scenario_order if s in scenario_styles
        ]
        legend_kwargs = dict(title='Mass function', frameon=False, fontsize=9, loc=loc)
        if bbox_anch_coords is not None:
            legend_kwargs['bbox_to_anchor'] = bbox_anch_coords
        ax.legend(handles=handles, **legend_kwargs)

def _extract(df, col):
    """Pull per-scenario value arrays from a dataframe column."""
    return {
        s: df.loc[df['scenario'] == s, col].to_numpy()
        for s in scenario_order
    }


if cgw_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot.')

# 1) nearest SMBHB distance — legend shows by default
fig, ax = plt.subplots(figsize=(9, 4.8))
_scenario_histogram(
    ax,
    scenario_vals=_extract(cgw_sim_df, 'nearest_D'),
    xlabel='Nearest SMBHB comoving distance $D$ [Mpc]',
    logx=True,
)
fig.tight_layout()
fig.savefig("figures/nearest_SMBHB.pdf", dpi=300, bbox_inches='tight')

# 2) loudest-by-CGW binary parameters — legend on first subplot only
fig, axes = plt.subplots(2, 2, figsize=(13, 8))
plot_cfg = [
    ('loudest_h0', '$h_0$',                          True),
    ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]',   True),
    ('loudest_D',  r'$D_{\rm{comov}}$ [Mpc]',         True),
    ('loudest_f',  '$f$ [Hz]',                         True),
]
for i, (ax, (col, label, use_logx)) in enumerate(zip(axes.ravel(), plot_cfg)):
    _scenario_histogram(
        ax,
        scenario_vals=_extract(cgw_sim_df, col),
        xlabel=label,
        logx=use_logx,
        show_legend=(i == 0),
    )

fig.suptitle('Loudest Binary (max CGW SNR) Properties', fontsize=14, y=1.01)
fig.tight_layout()
fig.savefig("figures/loudest_binary_parameters_CGW.pdf", dpi=300, bbox_inches='tight')

In [71]:
fig, ax = plt.subplots(figsize=(3.5, 2.8))

_scenario_histogram(
    ax,
    scenario_vals=_extract(cgw_sim_df, 'loudest_cgw_snr'),
    xlabel=r'CGW SNR',
    logx=True,
    show_legend=True,
    vline=5.0,
    loc='upper right',
    bbox_anch_coords = (0.5, -0.7)
)

# fig.tight_layout()
fig.savefig("figures/loudest_binary_parameters_CGW_SNR.pdf", dpi=300, bbox_inches='tight')

In [72]:
# Sky maps for loudest-by-CGW binaries per simulation, with pulsar positions overlaid.

import numpy as np
import matplotlib.pyplot as plt


def _wrap_to_aitoff_lon(ra_rad: np.ndarray) -> np.ndarray:
    # Convert RA in [0, 2pi) to Aitoff longitude in [-pi, pi].
    return ((ra_rad + np.pi) % (2.0 * np.pi)) - np.pi

#     def _get_ind_psr_ra_dec_rad(p):
#         """
#         Get RA and Dec in radians from a libstempo tempopulsar object,
#         handling both equatorial (RAJ/DECJ) and ecliptic (ELONG/ELAT) par files.
#         Tempo2 always converts internally, so param[param_raj].val[0] is always
#         populated — but from Python the safest route is to check which keys exist.
#         """
#         pars = set(p.pars(which='set'))

#         if 'RAJ' in pars and 'DECJ' in pars:
#             ra  = float(p['RAJ'].val)   # already in radians
#             dec = float(p['DECJ'].val)  # already in radians
#         elif 'ELONG' in pars and 'ELAT' in pars:
#             # ecliptic coords — convert via astropy
#             from astropy.coordinates import SkyCoord
#             import astropy.units as u
#             c = SkyCoord(lon=float(p['ELONG'].val) * u.rad,
#                          lat=float(p['ELAT'].val)  * u.rad,
#                          frame='geocentricmeanecliptic')
#             icrs = c.icrs
#             ra  = icrs.ra.rad
#             dec = icrs.dec.rad
#         else:
#             raise ValueError(f"Pulsar {p.name}: cannot find RAJ/DECJ or ELONG/ELAT in {pars}")

#         return ra, dec


#     def _get_pulsar_ra_dec(psrs_obj):
#         ras, decs = [], []
#         for p in psrs_obj:
#             ra, dec = _get_ind_psr_ra_dec_rad(p)
#             ras.append(ra)
#             decs.append(dec)
#         return np.asarray(ras), np.asarray(decs)


# def _try_load_pulsars_for_overlay():
#     if 'psrs_clean' in globals() and globals()['psrs_clean'] is not None:
#         return globals()['psrs_clean']
#     try:
#         from data_loader import load_pulsars, filter_pulsars_15yr
#         psrs_unfiltered = load_pulsars(verbose=False)
#         psrs_clean_local, _, _ = filter_pulsars_15yr(psrs_unfiltered, verbose=False)
#         return psrs_clean_local
#     except Exception as e:
#         print(f'Could not load pulsars for overlay: {e}')
#         return None


def _wrap_ra(ra_rad):
    """
    Shift RA to centre on 12h (pi) and flip so RA increases right-to-left,
    matching the NANOGrav/astronomical sky map convention.
    """
    # Shift so 12h is at centre, then wrap to [-pi, pi]
    ra_shifted = (ra_rad - np.pi) % (2 * np.pi)
    ra_shifted = np.where(ra_shifted > np.pi, ra_shifted - 2 * np.pi, ra_shifted)
    # Negate to flip RA direction (east to the left)
    return -ra_shifted

def _galactic_plane():
    """
    Convert galactic plane (b=0, l=0..360) to equatorial (RA, Dec) in radians
    without astropy, using the IAU galactic coordinate rotation matrix.
    NGP: RA=192.859508 deg, Dec=27.128336 deg
    Galactic zero-longitude node: 122.932 deg
    """
    # IAU constants
    ra_ngp  = np.deg2rad(192.859508)
    dec_ngp = np.deg2rad(27.128336)
    l_ncp   = np.deg2rad(122.932)      # l of north celestial pole

    l = np.linspace(0, 2 * np.pi, 1000)
    b = np.zeros(1000)                 # galactic plane: b = 0

    # Galactic -> equatorial rotation
    sin_dec = (np.sin(b) * np.sin(dec_ngp)
               + np.cos(b) * np.cos(dec_ngp) * np.sin(l_ncp - l))
    dec = np.arcsin(np.clip(sin_dec, -1, 1))

    cos_ra_minus_rangp = (np.cos(b) * np.cos(l_ncp - l)) / np.cos(dec)
    sin_ra_minus_rangp = (np.sin(b) * np.cos(dec_ngp)
                          - np.cos(b) * np.sin(dec_ngp) * np.sin(l_ncp - l)) / np.cos(dec)
    ra = (np.arctan2(sin_ra_minus_rangp, cos_ra_minus_rangp) + ra_ngp) % (2 * np.pi)

    ra  = _wrap_ra(ra)

    # sort and split at the wrap boundary to avoid a line across the map
    order = np.argsort(ra)
    ra, dec = ra[order], dec[order]
    # find big jumps and mask them so matplotlib doesn't connect the two sides
    gaps = np.where(np.abs(np.diff(ra)) > np.pi / 2)[0] + 1
    ra  = np.insert(ra.astype(float),  gaps, np.nan)
    dec = np.insert(dec.astype(float), gaps, np.nan)

    return ra, dec

if cgw_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot sky maps.')

sky_df = cgw_sim_df[
    np.isfinite(cgw_sim_df['loudest_ra']) & np.isfinite(cgw_sim_df['loudest_dec'])
].copy()
if sky_df.empty:
    raise RuntimeError('No valid RA/Dec values for loudest-by-CGW sources.')

data = np.load('data/pulsar_sky_locations.npz')
pulsar_ra  = _wrap_ra(data['ras'])
pulsar_dec = data['decs']

gal_ra, gal_dec = _galactic_plane()

# ── shared plot settings ────────────────────────────────────────────────────
RA_LABELS = ['18h', '16h', '14h', '12h', '10h', '8h', '6h', '4h', '2h', '0h', '22h']

def _style_skyax(ax):
    ax.set_xticklabels(RA_LABELS, fontsize=8, color='grey')
    ax.yaxis.set_tick_params(labelsize=8)
    ax.grid(True, alpha=0.25, linestyle='--', linewidth=0.5)

# ── Sky map 1: loudest binaries colored by scenario ─────────────────────────
fig = plt.figure(figsize=(3.5, 3.0))
ax  = fig.add_subplot(111, projection='aitoff')

for scenario in scenario_order:
    sub = sky_df[sky_df['scenario'] == scenario]
    if sub.empty:
        continue
    style = scenario_styles.get(scenario, {'color': '#888888'})
    ax.scatter(
        _wrap_ra(sub['loudest_ra'].to_numpy()),
        sub['loudest_dec'].to_numpy(),
        s=18,
        alpha=0.65,
        color=style['color'],
        edgecolors='none',
        label=scenario_label_map.get(scenario, scenario),
        zorder=3,
    )

if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=25, marker='*',
        color='black',
        alpha=0.9,
        label='Pulsars',
        zorder=4,
    )

# galactic plane

_style_skyax(ax)
ax.legend(loc='lower center', frameon=False, fontsize=8,
          bbox_to_anchor=(0.5, -0.68), ncol=1)
fig.tight_layout()
fig.savefig("figures/SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')

# ── Sky map 2: loudest binaries colored by CGW SNR ──────────────────────────
fig = plt.figure(figsize=(4.5, 4.0))
ax  = fig.add_subplot(111, projection='aitoff')

sc = ax.scatter(
    _wrap_ra(sky_df['loudest_ra'].to_numpy()),
    sky_df['loudest_dec'].to_numpy(),
    c=sky_df['loudest_cgw_snr'].to_numpy(),
    cmap='plasma',
    s=18,
    alpha=0.8,
    edgecolors='none',
    zorder=3,
)

if pulsar_ra.size > 0:
    ax.scatter(
        pulsar_ra, pulsar_dec,
        s=25, marker='*',
        color='white',
        edgecolors='black',
        linewidths=0.5,
        alpha=0.95,
        label='Pulsars',
        zorder=4,
    )


cbar = plt.colorbar(sc, ax=ax, pad=0.08, shrink=0.75, orientation='vertical')
cbar.set_label('Loudest binary CGW SNR', fontsize=9)
cbar.ax.tick_params(labelsize=8)

_style_skyax(ax)
if pulsar_ra.size > 0:
    ax.legend(loc='lower center', frameon=False, fontsize=8,
              bbox_to_anchor=(0.5, -0.18), ncol=2)
fig.tight_layout()
fig.savefig("figures/coloured_SNR_loudest_binary_CGW.pdf", dpi=300, bbox_inches='tight')

## Continuous GW (CGW) SNR Diagnostics with Threshold

This section mirrors the CGW diagnostics above, but only keeps the loudest CGW source in each simulation if its CGW SNR is at or above a configurable threshold. Change `CGW_SNR_THRESHOLD` in the next cell whenever you want to tighten or relax the cut.


In [73]:
CGW_SNR_THRESHOLD = 5.0


def build_cgw_threshold_simulation_table(sim_df: pd.DataFrame, snr_threshold: float) -> pd.DataFrame:
    """Filter the per-simulation CGW summary table to simulations above the SNR cut."""
    if sim_df.empty or 'loudest_cgw_snr' not in sim_df.columns:
        return pd.DataFrame()

    thresholded = sim_df.loc[sim_df['loudest_cgw_snr'] >= snr_threshold].copy()
    if thresholded.empty:
        return pd.DataFrame()

    thresholded['threshold_snr'] = float(snr_threshold)
    return thresholded.reset_index(drop=True)


def _threshold_extract(df, col):
    return {
        s: df.loc[df['scenario'] == s, col].to_numpy()
        for s in threshold_scenario_order
    }


def _threshold_scenario_histogram(ax, scenario_vals: dict, xlabel: str, logx: bool = False, show_legend: bool = True, vline: float = None, loc: str = 'upper left'):
    all_vals = np.concatenate([v for v in scenario_vals.values() if len(v) > 0])
    all_vals = all_vals[np.isfinite(all_vals)]
    if len(all_vals) == 0:
        return

    if logx:
        bins = np.logspace(np.log10(all_vals.min()), np.log10(all_vals.max()), 25)
    else:
        bins = np.linspace(all_vals.min(), all_vals.max(), 25)

    for s in threshold_scenario_order:
        vals = scenario_vals.get(s, np.array([]))
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            continue
        style = threshold_scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})
        ax.hist(
            vals,
            bins=bins,
            histtype='step',
            color=style['color'],
            linestyle=style['linestyle'],
            linewidth=style['linewidth'],
            label=s,
            density=False,
        )

    ax.set_xlabel(xlabel, fontsize=11)
    ax.set_ylabel('Counts', fontsize=11)
    ax.tick_params(labelsize=9)
    # ax.spines['top'].set_visible(False)
    # ax.spines['right'].set_visible(False)
    if logx:
        ax.set_xscale('log')
    if vline is not None:
        ax.axvline(vline, color='black', linestyle='--', linewidth=1.5)
    if show_legend:
        handles = [
            mlines.Line2D(
                [], [],
                color=threshold_scenario_styles[s]['color'],
                linestyle=threshold_scenario_styles[s]['linestyle'],
                linewidth=threshold_scenario_styles[s]['linewidth'],
                label=threshold_scenario_label_map[s],
            )
            for s in threshold_scenario_order if s in threshold_scenario_styles
        ]
        if vline is not None:
            handles.append(mlines.Line2D(
                [], [],
                color='black',
                linestyle='dashed',
                linewidth=1.5,
                label='Threshold',
            ))
        ax.legend(handles=handles, title='Mass function', frameon=False, fontsize=9, loc=loc)


cgw_threshold_sim_df = build_cgw_threshold_simulation_table(cgw_sim_df, CGW_SNR_THRESHOLD)
print('CGW simulations with thresholded top-source diagnostics:', len(cgw_threshold_sim_df))
if cgw_threshold_sim_df.empty:
    print('No per-simulation CGW diagnostics met the SNR threshold.')
else:
    threshold_scenario_order = [s for s in SCENARIOS if s in set(cgw_threshold_sim_df.get('scenario', []))]
    threshold_scenario_label_map = {s: scenario_label_map.get(s, s) for s in threshold_scenario_order}
    threshold_scenario_styles = {
        s: scenario_styles.get(s, {'color': '#888888', 'linestyle': '-', 'linewidth': 1.5})
        for s in threshold_scenario_order
    }

    display(
        cgw_threshold_sim_df.groupby('scenario', as_index=False)
        .agg(
            n_sims=('sim_key', 'nunique'),
            median_threshold_snr=('loudest_cgw_snr', 'median'),
            median_nearest_D=('nearest_D', 'median'),
        )
    )
    display(cgw_threshold_sim_df.head())


CGW simulations with thresholded top-source diagnostics: 108


,scenario,n_sims,median_threshold_snr,median_nearest_D
0,optimistic,108,7.976376,60.310461


,scenario,run_id,source_file,sim_index,sim_key,nearest_D,loudest_cgw_snr,loudest_h0,loudest_Mc,loudest_D,loudest_f,loudest_ra,loudest_dec,loudest_psi,loudest_iota,loudest_phi0,threshold_snr
0,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim000/summary.pkl.gz,0,optimistic:2026-05-20_optimistic:0,108.752007,6.535964,5.819182e-15,1.585873e+09,108.752007,4.420853e-09,6.093750,0.940918,2.691406,2.783203,1.345703,5.0
1,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim003/summary.pkl.gz,3,optimistic:2026-05-20_optimistic:3,88.730598,9.250917,9.982745e-16,4.109262e+09,11223.199219,4.999964e-09,5.402344,0.333740,0.145020,0.200562,2.697266,5.0
2,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim005/summary.pkl.gz,5,optimistic:2026-05-20_optimistic:5,45.356632,9.200004,3.752163e-16,2.315347e+09,11223.199219,3.782202e-09,4.281250,-0.389160,1.678711,0.564941,6.125000,5.0
3,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim007/summary.pkl.gz,7,optimistic:2026-05-20_optimistic:7,28.595421,7.603668,1.879759e-16,1.335515e+09,222.056183,2.486693e-09,4.167969,-0.545410,2.376953,1.773438,5.046875,5.0
4,optimistic,2026-05-20_optimistic,runs/2026-05-20_optimistic/sim014/summary.pkl.gz,14,optimistic:2026-05-20_optimistic:14,32.342430,11.604530,6.888683e-15,3.879299e+09,1029.243896,1.607298e-08,3.517578,0.484863,2.656250,2.994141,0.815918,5.0


In [74]:
# --- SNR threshold detection summary ---
# Count sims above threshold per scenario (already filtered in cgw_threshold_sim_df)
if not cgw_sim_df.empty:
    total_sims_df = (
        cgw_sim_df
        .groupby('scenario')['sim_key']
        .nunique()
        .rename('n_sims_total')
    )

    if not cgw_threshold_sim_df.empty and 'scenario' in cgw_threshold_sim_df.columns:
        above_threshold = (
            cgw_threshold_sim_df
            .groupby('scenario')['sim_key']
            .nunique()
            .rename('n_sims_above_threshold')
        )
    else:
        above_threshold = pd.Series(dtype=int, name='n_sims_above_threshold')

    threshold_summary = (
        total_sims_df
        .to_frame()
        .join(above_threshold, how='left')
        .fillna({'n_sims_above_threshold': 0})
        .assign(n_sims_above_threshold=lambda df: df['n_sims_above_threshold'].astype(int))
        .assign(detection_fraction=lambda df: df['n_sims_above_threshold'] / df['n_sims_total'])
        .reset_index()
    )

    # Preserve scenario ordering
    threshold_summary['scenario'] = pd.Categorical(
        threshold_summary['scenario'],
        categories=[s for s in SCENARIOS if s in threshold_summary['scenario'].values],
        ordered=True,
    )
    threshold_summary = threshold_summary.sort_values('scenario')

    print(f"\nSNR threshold: {CGW_SNR_THRESHOLD}")
    print("Simulations with at least one binary above SNR threshold, by scenario:\n")
    display(threshold_summary)
else:
    print('No `cgw_sim_df` available or it is empty.')



SNR threshold: 5.0
Simulations with at least one binary above SNR threshold, by scenario:



,scenario,n_sims_total,n_sims_above_threshold,detection_fraction
0,optimistic,338,108,0.319527
2,realistic,1,0,0.000000
1,pessimistic,1,0,0.000000


In [75]:
if cgw_threshold_sim_df.empty:
    raise RuntimeError('No CGW simulation table available to plot (all simulations below threshold).')

# 1) nearest SMBHB distance
fig, ax = plt.subplots(figsize=(9, 4.8))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'nearest_D'),
    xlabel=r'$D_{\rm{comov}}$ [Mpc]',
    logx=True,
)
fig.tight_layout()
fig.savefig("figures/nearest_SMBHB_threshold.pdf", dpi=300, bbox_inches='tight')

# 2) loudest-by-CGW binary parameters — legend on first subplot only
fig, axes = plt.subplots(2, 2, figsize=(9.0, 5.8))
plot_cfg = [
    ('loudest_h0', '$h_0$',                          True),
    ('loudest_Mc', r'$\mathcal{M}_c$ [$M_\odot$]',   True),
    ('loudest_D',  r'$D_{\rm{comov}}$ [Mpc]',         True),
    ('loudest_f',  '$f$ [Hz]',                         True),
]
for i, (ax, (col, label, use_logx)) in enumerate(zip(axes.ravel(), plot_cfg)):
    _threshold_scenario_histogram(
        ax,
        scenario_vals=_threshold_extract(cgw_threshold_sim_df, col),
        xlabel=label,
        logx=use_logx,
        show_legend=(i == 2),
        loc = 'upper left'
    )
# axes.ravel()[-1].axis('off')

fig.tight_layout()
fig.savefig("figures/loudest_binary_parameters_CGW_threshold.pdf", dpi=300, bbox_inches='tight')

# 3) CGW SNR distribution with threshold line
fig, ax = plt.subplots(figsize=(6.5, 4))
_threshold_scenario_histogram(
    ax,
    scenario_vals=_threshold_extract(cgw_threshold_sim_df, 'loudest_cgw_snr'),
    xlabel=r'CGW SNR',
    logx=True,
    show_legend=True,
    vline=CGW_SNR_THRESHOLD
)
fig.tight_layout()
fig.savefig("figures/loudest_binary_parameters_CGW_SNR_threshold.pdf", dpi=300, bbox_inches='tight')

In [76]:
# Sky maps for thresholded loudest-by-CGW binaries
threshold_sky_df = cgw_threshold_sim_df[
    np.isfinite(cgw_threshold_sim_df['loudest_ra']) & np.isfinite(cgw_threshold_sim_df['loudest_dec'])
].copy()

if not threshold_sky_df.empty:
    # ── Sky map 1: thresholded loudest binaries colored by scenario ─────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    for scenario in threshold_scenario_order:
        sub = threshold_sky_df[threshold_sky_df['scenario'] == scenario]
        if sub.empty:
            continue
        style = threshold_scenario_styles.get(scenario, {'color': '#888888'})
        ax.scatter(
            _wrap_ra(sub['loudest_ra'].to_numpy()),
            sub['loudest_dec'].to_numpy(),
            s=18,
            alpha=0.65,
            color=style['color'],
            edgecolors='none',
            label=threshold_scenario_label_map.get(scenario, scenario),
            zorder=3,
        )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='black',
            alpha=0.9,
            label='Pulsars',
            zorder=4,
        )

    _style_skyax(ax)
    ax.legend(loc='lower center', frameon=False, fontsize=8,
              bbox_to_anchor=(0.5, -0.18), ncol=len(threshold_scenario_order) + 2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f})'.format(CGW_SNR_THRESHOLD), fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')

    # ── Sky map 2: thresholded loudest binaries colored by CGW SNR ────────────────────
    fig = plt.figure(figsize=(11, 5.5))
    ax  = fig.add_subplot(111, projection='aitoff')

    sc = ax.scatter(
        _wrap_ra(threshold_sky_df['loudest_ra'].to_numpy()),
        threshold_sky_df['loudest_dec'].to_numpy(),
        c=threshold_sky_df['loudest_cgw_snr'].to_numpy(),
        cmap='plasma',
        s=18,
        alpha=0.8,
        edgecolors='none',
        zorder=3,
    )

    if pulsar_ra.size > 0:
        ax.scatter(
            pulsar_ra, pulsar_dec,
            s=25, marker='*',
            color='white',
            edgecolors='black',
            linewidths=0.5,
            alpha=0.95,
            label='Pulsars',
            zorder=4,
        )

    cbar = plt.colorbar(sc, ax=ax, pad=0.08, shrink=0.75, orientation='vertical')
    cbar.set_label('Loudest binary CGW SNR', fontsize=9)
    cbar.ax.tick_params(labelsize=8)

    _style_skyax(ax)
    if pulsar_ra.size > 0:
        ax.legend(loc='lower center', frameon=False, fontsize=8,
                  bbox_to_anchor=(0.5, -0.18), ncol=2)
    fig.suptitle('Sky Locations of Thresholded Loudest-by-CGW Binaries (SNR ≥ {:.1f}, coloured by CGW SNR)'.format(CGW_SNR_THRESHOLD), fontsize=12, y=1.01)
    fig.tight_layout()
    fig.savefig("figures/coloured_SNR_loudest_binary_CGW_threshold.pdf", dpi=300, bbox_inches='tight')
else:
    print("No valid RA/Dec values for thresholded loudest-by-CGW sources.")

## Multi-Simulation Population Comparisons

This section overlays binary frequency distributions from all simulations of a chosen scenario, filtering optionally by maximum redshift. Select the scenario and redshift cutoff below.

In [81]:
# ========== Configuration ==========
# Select which scenario to analyze
COMPARISON_SCENARIO = "pessimistic"  # Choose from: "optimistic", "pessimistic", "realistic"

# Toggle redshift filtering (set to None to include all redshifts, or specify max z)
MAX_REDSHIFT = None  # Set to a value like 2.0, 5.0, etc., or None for no cutoff

# Streaming parameters for memory efficiency
ALPHA_POPULATION = 0.08  # Transparency for each population line (lower = less clutter)
N_FREQ_BINS = 30         # Number of frequency bins if not supplied explicitly

# Candidate vertical lines, matching the visualisation.py interface
CANDIDATE_FREQUENCIES = None  # e.g. [1e-7, 2e-7] or None
CANDIDATE_LABELS = None       # e.g. ["Candidate 1", "Candidate 2"] or None
CANDIDATE_MASSES = None       # optional log10(Mtot/Msun) annotations for candidates

print("Configuration (Streaming plot_binaries_vs_frequency style):")
print(f"  Scenario: {COMPARISON_SCENARIO}")
print(f"  Max redshift: {MAX_REDSHIFT if MAX_REDSHIFT is not None else 'None (all)'}")
print(f"  Alpha per population: {ALPHA_POPULATION}")
print(f"  Frequency bins: {N_FREQ_BINS}")
print(f"  Candidate frequencies: {CANDIDATE_FREQUENCIES}")
print("\n✓ Will draw one step-line per population, with low alpha, on a single axes")

Configuration (Streaming plot_binaries_vs_frequency style):
  Scenario: pessimistic
  Max redshift: None (all)
  Alpha per population: 0.08
  Frequency bins: 30
  Candidate frequencies: None

✓ Will draw one step-line per population, with low alpha, on a single axes


In [ ]:
def streaming_plot_binaries_vs_frequency(
    files_list,
    scenario_name,
    max_z=None,
    mass_bins=None,
    freq_bins=None,
    candidate_frequencies=None,
    candidate_labels=None,
    candidate_masses=None,
    subset_name='Subset',
    n_freq_bins=30,
    alpha_population=0.08,
):
    """Stream a `plot_binaries_vs_frequency`-style figure without loading all populations.

    Each population is drawn as its own low-alpha step line, with the same mass-bin
    colors, candidate vertical lines, log-log axes, and legend handling as the
    reference implementation in visualisation.py.
    """
    from config import Msun

    if mass_bins is None:
        mass_bins = np.arange(7.5, 10.6, 0.5)

    def _population_mass_and_frequency(population):
        arrays = _population_to_arrays(population)
        if arrays is None or 'f' not in arrays:
            return None, None

        fgw = np.asarray(arrays['f'], dtype=float)
        fgw = fgw[np.isfinite(fgw)]
        if fgw.size == 0:
            return None, None

        if 'Mtot' in arrays and arrays['Mtot'] is not None:
            total_mass = np.asarray(arrays['Mtot'], dtype=float)
        elif 'Mc' in arrays and arrays['Mc'] is not None and 'q' in arrays and arrays['q'] is not None:
            Mc = np.asarray(arrays['Mc'], dtype=float)
            q = np.asarray(arrays['q'], dtype=float)
            total_mass = Mc * (1.0 + q) ** (6.0 / 5.0) / (q ** (3.0 / 5.0))
        else:
            total_mass = None

        if total_mass is not None:
            total_mass = np.asarray(total_mass, dtype=float)
            if total_mass.size != fgw.size:
                n = min(total_mass.size, fgw.size)
                total_mass = total_mass[:n]
                fgw = fgw[:n]
            mask = np.isfinite(total_mass) & np.isfinite(fgw)
            total_mass = total_mass[mask]
            fgw = fgw[mask]
            if total_mass.size == 0:
                return None, None

        return fgw, total_mass

    # Pass 1: determine frequency range without retaining populations.
    if freq_bins is None:
        f_min = np.inf
        f_max = 0.0
        n_populations_seen = 0

        for fp in files_list:
            if infer_scenario(fp) != scenario_name:
                continue
            try:
                payload = _load_payload(fp)
            except Exception:
                continue
            if not isinstance(payload, dict):
                continue
            pops = payload.get('populations', [])
            if not isinstance(pops, list):
                continue

            for entry in pops:
                if not isinstance(entry, dict):
                    continue
                population = entry.get('population')
                if population is None:
                    continue
                if max_z is not None:
                    population = _filter_population_by_redshift(population, max_z)
                fgw, _ = _population_mass_and_frequency(population)
                if fgw is None or fgw.size == 0:
                    continue
                f_min = min(f_min, float(np.nanmin(fgw)))
                f_max = max(f_max, float(np.nanmax(fgw)))
                n_populations_seen += 1

        if n_populations_seen == 0 or not np.isfinite(f_min) or not np.isfinite(f_max):
            print(f"No usable populations found for scenario '{scenario_name}'")
            return None

        freq_bins = np.logspace(np.log10(f_min), np.log10(f_max), n_freq_bins)

    bin_centers = 0.5 * (freq_bins[:-1] + freq_bins[1:])
    n_mass = len(mass_bins) - 1

    colors = [
        "#0072B2",
        "#E69F00",
        "#009E73",
        "#D55E00",
        "#CC79A7",
        "#56B4E9",
        "#000000",
    ]
    linestyles = ['-', '--']
    cand_colors = ['r', 'm', 'c', 'y']

    fig, ax = plt.subplots(figsize=(8, 6))

    # Stream and plot each population immediately.
    n_plotted = 0
    for fp in files_list:
        if infer_scenario(fp) != scenario_name:
            continue
        try:
            payload = _load_payload(fp)
        except Exception:
            continue
        if not isinstance(payload, dict):
            continue
        pops = payload.get('populations', [])
        if not isinstance(pops, list):
            continue

        for entry in pops:
            if not isinstance(entry, dict):
                continue
            population = entry.get('population')
            if population is None:
                continue
            if max_z is not None:
                population = _filter_population_by_redshift(population, max_z)

            fgw, total_mass = _population_mass_and_frequency(population)
            if fgw is None or fgw.size == 0:
                continue

            total_counts, _ = np.histogram(fgw, bins=freq_bins)
            ax.plot(
                bin_centers,
                np.where(total_counts > 0, total_counts, 0.1),
                color='k',
                linewidth=1.0,
                alpha=alpha_population,
                linestyle='-',
                drawstyle='steps-mid',
                label='Total' if n_plotted == 0 else None,
                zorder=1,
            )

            if total_mass is not None and total_mass.size == fgw.size:
                log_mass = np.log10(total_mass)
                for i in range(n_mass):
                    mask = (log_mass >= mass_bins[i]) & (log_mass < mass_bins[i + 1])
                    if not np.any(mask):
                        continue
                    counts, _ = np.histogram(fgw[mask], bins=freq_bins)
                    ax.plot(
                        bin_centers,
                        np.where(counts > 0, counts, 0.1),
                        color=colors[i % len(colors)],
                        linewidth=1.6,
                        alpha=alpha_population,
                        linestyle=linestyles[i % len(linestyles)],
                        drawstyle='steps-mid',
                        label=(r"$%.1f < \log_{10}\!\left(M_{\rm tot}/\mathrm{M}_\odot\right) < %.1f$" % (mass_bins[i], mass_bins[i + 1])) if n_plotted == 0 else None,
                        zorder=2,
                    )
            n_plotted += 1

    if n_plotted == 0:
        print(f"No populations found for scenario '{scenario_name}'")
        return None

    # Candidate frequencies match the reference interface.
    if candidate_frequencies is not None:
        if candidate_labels is None:
            candidate_labels = [f"Candidate {j + 1}" for j in range(len(candidate_frequencies))]
        if candidate_masses is None:
            candidate_masses = [None] * len(candidate_frequencies)

        for i, (f0, label, mass_val) in enumerate(zip(candidate_frequencies, candidate_labels, candidate_masses)):
            if mass_val is not None:
                label = (
                    rf"{label} "
                    rf"$\left[\log_{{10}}\!\left(M_{{\rm tot}}/M_\odot\right)={mass_val:.2f}\right]$"
                )
            ax.axvline(
                f0,
                color=cand_colors[i % len(cand_colors)],
                linestyle='-',
                lw=3,
                alpha=1,
                label=label,
                zorder=5,
            )

    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_xlabel('Gravitational Wave Frequency [Hz]', fontsize=12, fontweight='bold')
    ax.set_ylabel('Number of binaries', fontsize=12, fontweight='bold')
    ax.set_title(f'Binaries by GW frequency ({subset_name})', fontsize=13, fontweight='bold')

    from matplotlib.ticker import LogLocator
    ax.xaxis.set_major_locator(LogLocator(base=10))
    ax.xaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.yaxis.set_major_locator(LogLocator(base=10))
    ax.yaxis.set_minor_locator(LogLocator(base=10, subs=np.arange(2, 10) * 0.1))
    ax.tick_params(which='both', direction='in', top=True, right=True)

    ax.set_xlim(freq_bins[0], freq_bins[-1] * 1.2)
    ax.set_ylim(0.2, ax.get_ylim()[1])

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_color('black')
        spine.set_linewidth(1)

    handles, labels = ax.get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    ax.legend(by_label.values(), by_label.keys(), fontsize=9, frameon=True, edgecolor='black')

    plt.tight_layout()
    return fig


In [ ]:
# Execute the streaming comparison with the exact plot style from visualisation.py
print("\n" + "=" * 70)
print(f"Generating streaming multi-simulation frequency comparison for {COMPARISON_SCENARIO}...")
if MAX_REDSHIFT is not None:
    print(f"Redshift filter: z ≤ {MAX_REDSHIFT}")
print("=" * 70)

fig = streaming_plot_binaries_vs_frequency(
    result_files_CGW,
    scenario_name=COMPARISON_SCENARIO,
    max_z=MAX_REDSHIFT,
    mass_bins=np.arange(7.5, 10.6, 0.5),
    freq_bins=None,
    candidate_frequencies=CANDIDATE_FREQUENCIES,
    candidate_labels=CANDIDATE_LABELS,
    candidate_masses=CANDIDATE_MASSES,
    subset_name=COMPARISON_SCENARIO,
    n_freq_bins=N_FREQ_BINS,
    alpha_population=ALPHA_POPULATION,
)

if fig is not None:
    fname = f"figures/multi_sim_frequency_comparison_{COMPARISON_SCENARIO}"
    if MAX_REDSHIFT is not None:
        fname += f"_z{MAX_REDSHIFT}"
    fname += ".pdf"
    fig.savefig(fname, dpi=300, bbox_inches='tight')
    print(f"\n✓ Saved to {fname}")
    plt.close(fig)
else:
    print(f"\n✗ Could not generate plot for {COMPARISON_SCENARIO}")


Generating multi-simulation frequency comparison...
[PASS 1] Scanning 2398 files for frequency range...
  ✓ Found 800 populations in f ∈ [1.98e-09, 3.8e-07] Hz
[PASS 2] Computing mean histograms (sampling 50 individual curves)...
    Processed 50 simulations...
    Processed 100 simulations...
    Processed 150 simulations...
    Processed 200 simulations...
    Processed 250 simulations...
    Processed 300 simulations...
    Processed 350 simulations...
    Processed 400 simulations...
    Processed 450 simulations...
    Processed 500 simulations...


In [ ]:
# Summary printout: counts and min/max for CGW SNR histograms
import numpy as np

print('\n=== CGW SNR Summary ===')
if 'cgw_sim_df' in globals() and not cgw_sim_df.empty:
    col = 'loudest_cgw_snr'
    scenarios = sorted(cgw_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_sim_df.loc[cgw_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}': plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}': plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global SNR: no finite values to plot')
else:
    print('No `cgw_sim_df` available or it is empty.')

print('\n=== Thresholded CGW SNR Summary ===')
if 'cgw_threshold_sim_df' in globals() and not cgw_threshold_sim_df.empty:
    col = 'loudest_cgw_snr'
    scenarios = sorted(cgw_threshold_sim_df['scenario'].unique())
    total_vals = []
    for s in scenarios:
        arr = cgw_threshold_sim_df.loc[cgw_threshold_sim_df['scenario'] == s, col].to_numpy(dtype=float)
        finite = arr[np.isfinite(arr)]
        total_vals.append(finite)
        if finite.size:
            print(f"Scenario '{s}' (thresholded): plotted {finite.size} SNR values; min={finite.min():.3g}, max={finite.max():.3g}")
        else:
            print(f"Scenario '{s}' (thresholded): plotted 0 SNR values")
    all_vals = np.concatenate([a for a in total_vals if a.size > 0]) if total_vals else np.array([])
    if all_vals.size:
        print(f"Global thresholded SNR: n={all_vals.size}, min={all_vals.min():.3g}, max={all_vals.max():.3g}")
    else:
        print('Global thresholded SNR: no finite values to plot')
else:
    print('No `cgw_threshold_sim_df` available or it is empty.')